In [ ]:
%pip install xgboost

import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Cargar datos desde la Tabla Delta optimizada
df_spark = spark.table("workspace.default.paysim_features_delta")

# Convertir a Pandas (solo las variables relevantes para modelado)
feature_cols = [
    "step", "amount", "old_balance_orig", "new_balance_orig", 
    "old_balance_dest", "new_balance_dest", 
    "error_balance_orig", "error_balance_dest"
]
target_col = "is_fraud"

df_pd = df_spark.select(feature_cols + [target_col]).toPandas()

X = df_pd[feature_cols]
y = df_pd[target_col]

# 2. Dividir Dataset (Train/Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Iniciar Experimento con MLflow Tracking
mlflow.autolog() # Registra automáticamente parámetros, métricas y el modelo

with mlflow.start_run(run_name="XGBoost_Fraud_Detection_Baseline"):
    # Calcular scale_pos_weight para desbalance de clases
    ratio = float(len(y_train[y_train == 0])) / len(y_train[y_train == 1])
    
    # Entrenar modelo XGBoost
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=ratio,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    # Evaluar
    y_pred = model.predict(X_test)
    print("=== Reporte de Clasificación ===")
    print(classification_report(y_test, y_pred))

print("✅ Modelo entrenado y registrado exitosamente en MLflow.")